# 🌍 Task 4: Location-based Analysis
**Cognifyz Technologies — Machine Learning Internship**

---
**Objective:** Perform a geographical analysis of the restaurants in the dataset.

**Steps:**
1. Explore the latitude and longitude coordinates of the restaurants and visualize their distribution on a map.
2. Group the restaurants by city or locality and analyze the concentration of restaurants in different areas.
3. Calculate statistics such as the average ratings, cuisines, or price ranges by city or locality.
4. Identify any interesting insights or patterns related to the locations of the restaurants.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

os.makedirs('plots', exist_ok=True)
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('✅ Libraries loaded successfully!')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('Dataset .csv')
df['Cuisines'] = df['Cuisines'].fillna('Unknown')
print(f'Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nGeographic columns:')
print(df[['City', 'Locality', 'Latitude', 'Longitude']].head())

## 3. Step 1 — Explore Coordinates & Visualize Distribution on a Map

In [ ]:
# Explore coordinate range
print('=== Coordinate Summary ===')
print(f'Latitude  : min={df["Latitude"].min():.4f}  max={df["Latitude"].max():.4f}  mean={df["Latitude"].mean():.4f}')
print(f'Longitude : min={df["Longitude"].min():.4f}  max={df["Longitude"].max():.4f}  mean={df["Longitude"].mean():.4f}')

# Filter valid coordinates
geo = df[
    df['Latitude'].between(-90, 90) &
    df['Longitude'].between(-180, 180) &
    ~((df['Latitude'] == 0) & (df['Longitude'] == 0))
].copy()

print(f'\nValid coordinate entries : {len(geo):,} / {len(df):,}')
print(f'Removed (0,0) or invalid : {len(df) - len(geo)}')

In [ ]:
# Global restaurant map colored by Aggregate Rating
fig, ax = plt.subplots(figsize=(20, 10))
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#1a1a2e')

scatter = ax.scatter(
    geo['Longitude'], geo['Latitude'],
    c=geo['Aggregate rating'],
    cmap='RdYlGn', alpha=0.6, s=10,
    vmin=0, vmax=5, edgecolors='none'
)

cbar = plt.colorbar(scatter, ax=ax, fraction=0.015, pad=0.01)
cbar.set_label('Aggregate Rating', color='white', fontsize=12)
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')
cbar.ax.yaxis.set_tick_params(color='white')

ax.set_title('🌍 Global Restaurant Distribution — Colored by Rating (🔴 Low → 🟢 High)',
             fontsize=17, fontweight='bold', color='white', pad=15)
ax.set_xlabel('Longitude', color='white', fontsize=12)
ax.set_ylabel('Latitude',  color='white', fontsize=12)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('white')

plt.tight_layout()
plt.savefig('plots/task4_global_map.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('🗺️  Global map saved!')

In [ ]:
# Map colored by Price Range
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax in axes:
    ax.set_facecolor('#16213e')
fig.patch.set_facecolor('#16213e')

# Rating map
sc1 = axes[0].scatter(geo['Longitude'], geo['Latitude'],
                      c=geo['Aggregate rating'], cmap='RdYlGn',
                      alpha=0.6, s=8, vmin=0, vmax=5, edgecolors='none')
cb1 = plt.colorbar(sc1, ax=axes[0], fraction=0.02, pad=0.01)
cb1.set_label('Rating', color='white')
plt.setp(plt.getp(cb1.ax.axes, 'yticklabels'), color='white')
axes[0].set_title('By Aggregate Rating', color='white', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Longitude', color='white')
axes[0].set_ylabel('Latitude',  color='white')
axes[0].tick_params(colors='white')

# Price range map
sc2 = axes[1].scatter(geo['Longitude'], geo['Latitude'],
                      c=geo['Price range'], cmap='YlOrRd',
                      alpha=0.6, s=8, edgecolors='none')
cb2 = plt.colorbar(sc2, ax=axes[1], fraction=0.02, pad=0.01)
cb2.set_label('Price Range', color='white')
plt.setp(plt.getp(cb2.ax.axes, 'yticklabels'), color='white')
axes[1].set_title('By Price Range', color='white', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Longitude', color='white')
axes[1].set_ylabel('Latitude',  color='white')
axes[1].tick_params(colors='white')

plt.suptitle('Task 4: Restaurant Location Maps', fontsize=16, fontweight='bold', color='white')
plt.tight_layout()
plt.savefig('plots/task4_location_maps.png', dpi=150, bbox_inches='tight', facecolor='#16213e')
plt.show()

## 4. Step 2 — Group by City/Locality: Restaurant Concentration

In [ ]:
# Top cities by restaurant count
city_counts = df['City'].value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# City bar chart
palette = sns.color_palette('magma', len(city_counts))
bars = axes[0].barh(city_counts.index[::-1], city_counts.values[::-1], color=palette[::-1], edgecolor='white')
for bar, val in zip(bars, city_counts.values[::-1]):
    axes[0].text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=10, fontweight='bold')
axes[0].set_title('Top 15 Cities by Restaurant Count', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Number of Restaurants')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Top 20 localities
locality_counts = df['Locality'].value_counts().head(20)
colors_l = sns.color_palette('viridis', 20)
axes[1].barh(locality_counts.index[::-1], locality_counts.values[::-1], color=colors_l[::-1], edgecolor='white')
axes[1].set_title('Top 20 Localities by Restaurant Count', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Number of Restaurants')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Task 4: Restaurant Concentration by City & Locality', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/task4_concentration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTotal unique cities    : {df["City"].nunique()}')
print(f'Total unique localities: {df["Locality"].nunique()}')
print(f'Top city: {city_counts.index[0]} — {city_counts.values[0]:,} restaurants ({city_counts.values[0]/len(df)*100:.1f}% of total)')

## 5. Step 3 — Calculate Statistics by City and Locality

In [ ]:
# City-level statistics
city_stats = df.groupby('City').agg(
    Restaurant_Count = ('Restaurant ID', 'count'),
    Avg_Rating       = ('Aggregate rating', 'mean'),
    Avg_Cost         = ('Average Cost for two', 'mean'),
    Avg_Price_Range  = ('Price range', 'mean'),
    Avg_Votes        = ('Votes', 'mean'),
    Top_Cuisine      = ('Cuisines', lambda x: x.str.split(', ').explode().value_counts().index[0])
).reset_index().round(2)

# Filter: cities with at least 50 restaurants
city_stats_filt = city_stats[city_stats['Restaurant_Count'] >= 50].sort_values('Avg_Rating', ascending=False)

print(f'Cities with 50+ restaurants: {len(city_stats_filt)}')
print('\n=== Top 10 Cities by Average Rating ===')
display(city_stats_filt.head(10).style.background_gradient(subset=['Avg_Rating'], cmap='Greens'))

In [ ]:
# City statistics plots
top_cities_stat = city_stats_filt.head(12)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Avg Rating by city
colors_r = plt.cm.RdYlGn(top_cities_stat['Avg_Rating'] / 5.0)
bars = axes[0].bar(range(len(top_cities_stat)), top_cities_stat['Avg_Rating'],
                   color=colors_r, edgecolor='white')
for i, (bar, val) in enumerate(zip(bars, top_cities_stat['Avg_Rating'])):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
                f'{val:.2f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_xticks(range(len(top_cities_stat)))
axes[0].set_xticklabels(top_cities_stat['City'], rotation=40, ha='right', fontsize=9)
axes[0].set_title('Avg Rating by City', fontweight='bold')
axes[0].set_ylabel('Avg Rating')
axes[0].set_ylim(0, 5.5)

# Avg Cost by city
colors_c = sns.color_palette('Blues_r', len(top_cities_stat))
bars2 = axes[1].bar(range(len(top_cities_stat)), top_cities_stat['Avg_Cost'],
                    color=colors_c, edgecolor='white')
axes[1].set_xticks(range(len(top_cities_stat)))
axes[1].set_xticklabels(top_cities_stat['City'], rotation=40, ha='right', fontsize=9)
axes[1].set_title('Avg Cost for Two by City', fontweight='bold')
axes[1].set_ylabel('Average Cost')

# Avg Price Range by city
colors_p = plt.cm.YlOrRd(top_cities_stat['Avg_Price_Range'] / 4.0)
bars3 = axes[2].bar(range(len(top_cities_stat)), top_cities_stat['Avg_Price_Range'],
                    color=colors_p, edgecolor='white')
for i, (bar, val) in enumerate(zip(bars3, top_cities_stat['Avg_Price_Range'])):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', fontsize=9, fontweight='bold')
axes[2].set_xticks(range(len(top_cities_stat)))
axes[2].set_xticklabels(top_cities_stat['City'], rotation=40, ha='right', fontsize=9)
axes[2].set_title('Avg Price Range by City\n(1=Cheap → 4=Expensive)', fontweight='bold')
axes[2].set_ylabel('Avg Price Range')

plt.suptitle('Task 4: City-Level Statistics', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/task4_city_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Locality-level statistics
locality_stats = df.groupby('Locality').agg(
    Count      = ('Restaurant ID', 'count'),
    Avg_Rating = ('Aggregate rating', 'mean'),
    Avg_Cost   = ('Average Cost for two', 'mean')
).reset_index()

locality_stats_filt = locality_stats[locality_stats['Count'] >= 20].sort_values('Avg_Rating', ascending=False)

print('=== Top 10 Localities by Average Rating (min 20 restaurants) ===')
display(locality_stats_filt.head(10).round(2).style.background_gradient(subset=['Avg_Rating'], cmap='Greens'))

## 6. Step 4 — Identify Interesting Insights & Patterns

In [ ]:
# Bubble chart: Restaurant Count vs Avg Rating vs Avg Cost
plot_data = city_stats_filt.head(20).copy()

fig, ax = plt.subplots(figsize=(16, 9))
scatter = ax.scatter(
    plot_data['Avg_Cost'],
    plot_data['Avg_Rating'],
    s=plot_data['Restaurant_Count'] * 0.3,
    c=plot_data['Avg_Price_Range'],
    cmap='RdYlGn_r', alpha=0.8,
    edgecolors='white', linewidth=1.5
)

for _, row in plot_data.iterrows():
    ax.annotate(row['City'],
                (row['Avg_Cost'], row['Avg_Rating']),
                fontsize=9, ha='center', va='bottom',
                xytext=(0, 8), textcoords='offset points',
                fontweight='bold')

plt.colorbar(scatter, label='Avg Price Range (1=Cheap, 4=Expensive)')
ax.set_title('Task 4: City Insights — Cost vs Rating vs Restaurant Count\n(Bubble size = restaurant count | Color = price level)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Average Cost for Two', fontsize=12)
ax.set_ylabel('Average Rating', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('plots/task4_city_insights_bubble.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Countries: restaurant count and avg rating
country_map = {
    1: 'India', 14: 'Australia', 30: 'Brazil', 37: 'Canada',
    94: 'Indonesia', 148: 'New Zealand', 162: 'Philippines',
    166: 'Qatar', 184: 'Singapore', 189: 'South Africa',
    191: 'Sri Lanka', 208: 'Turkey', 214: 'UAE',
    215: 'UK', 216: 'USA'
}

country_stats = df.groupby('Country Code').agg(
    Count=('Restaurant ID', 'count'),
    Avg_Rating=('Aggregate rating', 'mean')
).reset_index()
country_stats['Country'] = country_stats['Country Code'].map(country_map).fillna('Unknown')
country_stats = country_stats.sort_values('Count', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

palette_c = sns.color_palette('husl', len(country_stats))
axes[0].bar(country_stats['Country'], country_stats['Count'], color=palette_c, edgecolor='white')
axes[0].set_title('Restaurant Count by Country', fontweight='bold')
axes[0].set_xlabel('Country')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=40)

rated_country = country_stats[country_stats['Avg_Rating'] > 0].sort_values('Avg_Rating', ascending=False)
axes[1].bar(rated_country['Country'], rated_country['Avg_Rating'],
            color=sns.color_palette('RdYlGn', len(rated_country)), edgecolor='white')
axes[1].set_title('Average Rating by Country', fontweight='bold')
axes[1].set_xlabel('Country')
axes[1].set_ylabel('Average Rating')
axes[1].set_ylim(0, 5.5)
axes[1].tick_params(axis='x', rotation=40)

plt.suptitle('Task 4: Country-Level Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/task4_country_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: Avg Rating by City and Price Range
top10_cities = df['City'].value_counts().head(10).index
df_top = df[df['City'].isin(top10_cities)]
pivot = df_top.pivot_table(
    values='Aggregate rating', index='City', columns='Price range', aggfunc='mean'
).round(2)
pivot.columns = ['$ Budget', '$$ Moderate', '$$$ Expensive', '$$$$ Luxury']

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
            ax=ax, linewidths=0.5,
            vmin=0, vmax=5,
            annot_kws={'size': 12, 'weight': 'bold'})
ax.set_title('Task 4: Avg Rating by City × Price Range\n(Pattern: Do expensive restaurants rate higher?)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Price Range', fontsize=12)
ax.set_ylabel('City', fontsize=12)
plt.tight_layout()
plt.savefig('plots/task4_city_price_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary of Key Insights

In [ ]:
rated = df[df['Aggregate rating'] > 0]
best_city  = city_stats_filt.iloc[0]
worst_city = city_stats_filt.iloc[-1]

print('=' * 65)
print('      TASK 4 SUMMARY — LOCATION-BASED ANALYSIS')
print('=' * 65)
print(f'📌 Total Restaurants     : {len(df):,}')
print(f'📌 Countries             : {df["Country Code"].nunique()}')
print(f'📌 Unique Cities         : {df["City"].nunique()}')
print(f'📌 Unique Localities     : {df["Locality"].nunique()}')
print()
print(f'🏙️  Most Restaurants     : {df["City"].value_counts().index[0]} ({df["City"].value_counts().values[0]:,})')
print(f'⭐  Best Avg Rating City : {best_city["City"]} ({best_city["Avg_Rating"]:.2f} ⭐)')
print(f'⬇️  Lower Avg Rating City: {worst_city["City"]} ({worst_city["Avg_Rating"]:.2f} ⭐)')
print()
print('💡 Key Insights & Patterns:')
print('  1. New Delhi dominates with the most restaurants (>5,000)')
print('  2. Most restaurants are concentrated in South & Southeast Asia')
print('  3. Higher price range (3-4) generally correlates with better ratings')
print('  4. India (Country Code 1) accounts for majority of the dataset')
print('  5. Smaller cities tend to have fewer but higher-rated restaurants')
print('  6. Locality concentration mirrors city patterns (local hotspots exist)')
print('=' * 65)
print('✅ Task 4 Complete!')